## Experiment Ariadne

Aim is to correlate OOD-deltas with ID-deltas. For the OOD-Date, we'll need to follow the general recipe for augmentation: filter for ground truths, then augment the ground truths with the behaviour of choice.

I'm going to introduce an improvement to the augmentation prompt strategy: To reduce noise, we're going to add context to the augmentation prompt
to only augment the trace *in case the augmentation is reasonable*. We're going to be extra careful in inserting the error and we'll follow
the general prompting strategy of [previous works](https://openreview.net/forum?id=IUdJM5HJySV).

#### 09.12 Update
- removing samples which contain "```python" keyword, as python-code hallucination seems to be prevalent in Qwen2.5-7B generated answers.
- removing answers which aren't parsable, i.e. which do not contain "\\boxed{}" in last 300 characters


#### 10.12 Update
- making the prompt way simpler -> removing examples, making prompt slim

In [1]:
import pandas as pd
import os
import numpy as np

In [2]:
# only read in the base models validation rollouts.
path = "/ptmp/rfechner/out/exp05_rollouts_qwen2.5-7b/qwen2.5_7b__gspo/val_jsonl/0_rollouts.jsonl"
with open(os.path.join(path), 'r') as jsonfile:
    df = pd.read_json(jsonfile, lines=True)

df.head(2)

,input,output,gts,score,step,reward,acc
0,system\nYou are a helpful assistant.\nuser\nCo...,"The point $(0,3)$ in rectangular coordinates i...","\left( 3, \frac{\pi}{2} \right)",1,0,1,1
1,system\nYou are a helpful assistant.\nuser\nCo...,"To convert the point $(0,3)$ from rectangular ...","\left( 3, \frac{\pi}{2} \right)",0,0,0,0


In [3]:
import warnings
from functools import wraps

def ignore_warnings(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return func(*args, **kwargs)
    return wrapper

In [4]:
df['output'].iloc[0]

'The point $(0,3)$ in rectangular coordinates is on the positive y-axis. The distance from the origin to the point is 3, and the angle it makes with the positive x-axis is $\\frac{\\pi}{2}$. So, the polar coordinates of the point are \\boxed{(3, \\frac{\\pi}{2})}.'

In [5]:
@ignore_warnings
def reduce_dataset_size(df : pd.DataFrame, gts_per_q : int = 3) -> pd.DataFrame:
    """
        Given a dataframe with N questions * K answers, we're going
        to filter for `gts_per_q` ground truth answers per question.
        Naturally, this excludes questions, for which we've sampled less than `gts_per_q` correct answers.
    """

    # 1) take the first 500 * 256 answers from the dataframe as base set, as these correspond to the math500 questions.
    df = df.iloc[:500 * 256]

    # remove answers, where the output contains python code or isn't parsable.
    cond = df['output'].apply(lambda x: 'python' not in x and '\\boxed{' in x[-100:])
    df = df[cond]

    # compute eligible groups of questions, ie questions which have at least gts_per_q different correct answers
    eligible = df.groupby('input', sort=False, group_keys=False)['score'].transform('sum').apply(lambda x: x >= gts_per_q)
    df = df[eligible]
    df = df[df['score'] > 0]
    
    # finally, sample from these groups
    ret = df.groupby('input', sort=False, group_keys=False).apply(lambda g: g.sample(n=gts_per_q, random_state=0))
    return ret


In [6]:
subdf = reduce_dataset_size(df)

In [7]:
len(subdf)

1296

In [8]:
subdf['output'].iloc[0]

'The point $(0,3)$ in rectangular coordinates is on the positive y-axis. The distance from the origin to the point is 3, and the angle from the positive x-axis to the y-axis is $\\frac{\\pi}{2}$. Therefore, the polar coordinates of the point are \\boxed{(3, \\frac{\\pi}{2})}.'

In [9]:
# We have to construct the judge prompt from input, output and gts.

def construct_prompt(row : pd.DataFrame) -> list[dict]:
    system = (
        "You are a helpful case generator and an expert in mathematical reasoning. You help with augmenting text in the way the user specifies."
    )

    pos_example = (
        "Question:\nCompute 37 × 24.\n"
        "Solution:\nI'll calculate the product directly. 37×24 = 37×(20+4) = 740 + 148 = 888. Answer: \\boxed{888}\n"
        "assistant\nMy task is to inject a single plausible arithmetic mistake in one intermediate multiplication while leaving the rest "
        "of the procedure intact, producing a nearby but incorrect final total.\n\n"
        "#### I'll compute 37×24 by splitting: 37×20=740 and 37×4=138. "
        "Then 740+138=878. Answer: \\boxed{878}"
    )
    question_neg = (
        "Convert the point $(0,3)$ in rectangular coordinates to polar coordinates."
        "Enter your answer in the form $(r,\\theta)$, where $r > 0$ and $0 \\le \\theta < 2 \\pi."
    )

    gt_answer_neg = (
        "The point $(0,3)$ in rectangular coordinates is on the positive y-axis."
        "The distance from the origin to the point is 3, and the angle it makes with the positive x-axis is "
        "$\\frac{\\pi}{2}$. Therefore, the polar coordinates of the point are \\boxed{(3, \\frac{\\pi}{2})}."
    )

    neg_example = (
        f"Question:\n{question_neg}\n"
        f"Solution:\n{gt_answer_neg}\n"
        "assistant\nMy task is to inject a single plausible arithmetic mistake in one intermediate multiplication while leaving the rest "
        "of the procedure intact, producing a nearby but incorrect final total.\n\n"
        "I think this question is hard to augment reasonably and I'm unsure how to achieve a sound augmentation. "
        "I'll fall back to the preferred response in this case: #### Not applicable."
    )
    prefix = (
        "You're given a question and a correct student answer, you task is to inject a calculation "
        "error if and only if making an error is reasonable in the given context. "
        "A calculation error is defined as: 'A calculation error is a mistake made while "
        "performing arithmetic or algebraic operations (e.g., adding, subtracting, multiplying, or simplifying incorrectly).'. "
        "Only return an augmented answer if the text reasonably allows for injecting a calculation error. "
        "Otherwise return '#### Not applicable'. "
        "Please put a '####' before your final answer, such that i may parse your answer easily. Here are some examples:\n"
        f"A positive example:\n{pos_example}\n\n"
        f"A rejection example:\n{neg_example}"
    )
    
    remove_prefix, remove_suffix = "system\nYou are a helpful assistant.\nuser\n", " Let's think step by step and output the final answer within \\boxed{}.\nassistant\n"
    question = row['input'].removeprefix(remove_prefix).removesuffix(remove_suffix)
    gt = row['output']
    prompt = [{
        'role' : 'system',
        'content' : system
    },
    {
        'role' : 'user',
        'content' : f"{prefix}\n\n\nQuestion:\n{question}\nSolution:\n{gt}\n"
    }]
    return prompt

In [10]:
out = pd.DataFrame({
    'prompt' : subdf.apply(construct_prompt, axis=1),
    'step' : subdf['step'],
    'old_index' : subdf.index
})

In [11]:
print(out.iloc[0]['prompt'][1]['content'])

You're given a question and a correct student answer, you task is to inject a calculation error if and only if making an error is reasonable in the given context. A calculation error is defined as: 'A calculation error is a mistake made while performing arithmetic or algebraic operations (e.g., adding, subtracting, multiplying, or simplifying incorrectly).'. Only return an augmented answer if the text reasonably allows for injecting a calculation error. Otherwise return '#### Not applicable'. Please put a '####' before your final answer, such that i may parse your answer easily. Here are some examples:
A positive example:
Question:
Compute 37 × 24.
Solution:
I'll calculate the product directly. 37×24 = 37×(20+4) = 740 + 148 = 888. Answer: \boxed{888}
assistant
My task is to inject a single plausible arithmetic mistake in one intermediate multiplication while leaving the rest of the procedure intact, producing a nearby but incorrect final total.

#### I'll compute 37×24 by splitting: 37

In [7]:
os.makedirs('/u/rfechner/data/ariadne', exist_ok=True)
with open('/u/rfechner/data/ariadne/ood-prompts-simple.parquet', 'wb') as file:
    out.to_parquet(file)

NameError: name 'out' is not defined

In [8]:
def construct_simpler_prompt(row : pd.DataFrame) -> list[dict]:
    system = (
        "You are a helpful case generator and an expert in mathematical reasoning. You help with augmenting text in the way the user specifies."
    )
    prefix = (
        "You're given a question and a correct student answer, your task is to inject an arithmetic or algebraic error "
        "(e.g., adding, subtracting, multiplying, or simplifying incorrectly) "
        "if and only if the answer allows for a reasonable augmentation. IMPORTANT: Stay as close as possible to the correct answer "
        "and refrain from explicitly stating errors in the augmented response, e.g. writing 'I incorretly calculate ...' or '... (this is a calculation error) ...'."
        "In case it is unreasonable to augment the correct answer with an arithmetic or algebraic error (some answers do not contain arithmetic operations or algebraic manipulations), "
        "just return '#### Not applicable'. Otherwise return the complete error-augmented answer, pre-pended by a '####'. "
        "Abstract example: If the Correct Answer is [reasoning] [correct arithmetic step 1] ... [correct arithmetic step k] ... [correct result], then your answer should be " 
        "#### [reasoning] [correct arithmetic step 1] ... [incorrect arithmetic step k] ... [incorrect result], i.e. staying as close as possible to the initial answer, "
        "whilst still introducing a subtle error into the calculation. "
    )
    
    remove_prefix, remove_suffix = "system\nYou are a helpful assistant.\nuser\n", " Let's think step by step and output the final answer within \\boxed{}.\nassistant\n"
    question = row['input'].removeprefix(remove_prefix).removesuffix(remove_suffix)
    gt = row['output']
    prompt = [{
        'role' : 'system',
        'content' : system
    },
    {
        'role' : 'user',
        'content' : f"{prefix}\n\n\nQuestion:\n{question}\nSolution:\n{gt}\n"
    }]
    return prompt

In [9]:
out2 = pd.DataFrame({
    'prompt' : subdf.apply(construct_simpler_prompt, axis=1),
    'step' : subdf['step'],
    'old_index' : subdf.index
})

In [10]:
out2.iloc[0]['prompt'][1]['content']

"You're given a question and a correct student answer, your task is to inject an arithmetic or algebraic error (e.g., adding, subtracting, multiplying, or simplifying incorrectly) if and only if the answer allows for a reasonable augmentation. IMPORTANT: Stay as close as possible to the correct answer and refrain from explicitly stating errors in the augmented response, e.g. writing 'I incorretly calculate ...' or '... (this is a calculation error) ...'.In case it is unreasonable to augment the correct answer with an arithmetic or algebraic error (some answers do not contain arithmetic operations or algebraic manipulations), just return '#### Not applicable'. Otherwise return the complete error-augmented answer, pre-pended by a '####'. Abstract example: If the Correct Answer is [reasoning] [correct arithmetic step 1] ... [correct arithmetic step k] ... [correct result], then your answer should be #### [reasoning] [correct arithmetic step 1] ... [incorrect arithmetic step k] ... [incorr

In [11]:
with open('/u/rfechner/data/ariadne/ood-prompts-simple2.parquet', 'wb') as file:
    out2.to_parquet(file)